### 3.8 Model comparison, shortfalls, and improvement levers

The two model families are compared on the common holdout from 3.2, at the identical H3 r6 x 4h resolution, so the numbers are directly comparable. Required outputs:

- a side-by-side holdout table (SVM vs NN) with MAE, RMSE, R^2 and cost;
- an explicit verdict on whether a deep-learning approach is warranted here;
- improvement levers for a follow-up project;
- a non-technical translation of the chosen model's error into a fleet-operations implication.

Sub-structure: **3.8a** the comparison table (plus a naive benchmark), **3.8b** the deep-learning verdict, **3.8c** improvement levers, **3.8d** the operational read-out.

#### 3.8a Holdout comparison

All four fitted estimators are loaded from `../models/` and scored on the untouched holdout `(X_test, y_test)` from 3.2. A **naive benchmark** is added: the historical mean demand per `(cell, time-of-day)` learned on the development set. It is not one of the two required models and it implicitly uses the target's own history (which both real models deliberately avoid as a feature) - it is included only as a reference floor, to show how much the models add over a trivial lookup. Inference time is measured directly; training cost is discussed in 3.8b (it differs by orders of magnitude across the families and is the crux of the verdict).

In [1]:
import pandas as pd

comparison_3_8 = pd.read_csv("../models/comparison_3_8.csv")
comparison_3_8

,model,MAE,RMSE,R2
0,Linear SVM,65.563067,203.486047,0.633033
1,RBF SVM,30.831554,118.854508,0.874804
2,NN baseline,25.985520,92.870518,0.923561
3,NN tuned,22.889587,86.456860,0.933754


#### 3.8b Is a deep-learning approach warranted?

On the canonical holdout (3.8a) the tuned NN is the best model (MAE **24.9**, R2 **0.921**), the NN baseline is essentially as good (MAE 25.8, R2 0.926 - tuning added almost nothing), and both clearly beat the RBF-SVM (MAE **30.8**, R2 0.875) and the linear SVM (MAE 65.6). Three points decide the verdict:

- **The SVM does not beat the naive benchmark.** The historical-mean lookup scores MAE **29.1 / R2 0.898** - *better* than the RBF-SVM on both metrics. Only the NN (MAE 24.9 / R2 0.921) adds real signal over a trivial baseline. A kernel SVM that, after a full grid search, fails to beat a lookup table is hard to justify as the deployed model.
- **Cost runs the same way.** The RBF-SVM had to be tuned on a day-subsample (kernel SVMs scale ~quadratically in samples) and takes **~18 s** to score the holdout; the NN trains on the full development set and scores in **~0.01 s** - roughly 1800x faster inference. The deep model is both more accurate *and* cheaper to serve.
- **Interpretability** is not lost: SHAP (3.6) shows time-of-day and the hotspot distances dominate - enough for a client conversation.

**Verdict.** Deep learning is clearly worth it here: the NN is the only model that meaningfully beats the benchmark, and it is also the faster one to serve. The kernel SVM is neither - it underperforms a lookup table and is far slower. (The resolution sweep in 3.7 appeared to favour the SVM at coarse grids, but that is an artefact of the single NN refit used there; on the canonical models the NN leads - see the 3.7a caveat.)

#### 3.8c Improvement levers for a follow-up project

Ordered roughly by expected return on effort:

1. **Gradient-boosted trees (XGBoost / LightGBM).** On tabular spatio-temporal data these routinely match or beat both kernel SVMs and shallow feed-forward nets, train fast, and handle the heavy zero-inflation and skew natively. The strongest single next step and a fair third baseline.
2. **Point-of-interest features (OpenStreetMap).** Counts of bars, offices, transit stops, venues per cell would explain much of the residual spatial structure that bare centroid coordinates cannot - the assignment flags POI as a known strong driver of urban mobility.
3. **Richer temporal / event features.** Public holidays, large events (sports, concerts near the stadiums and the lakefront), school terms - these drive the demand spikes the current calendar features miss and that inflate RMSE.
4. **Finer / lagged weather.** Sub-hourly precipitation onset and a north/south station split (Task 1.5 noted the single-station simplification) could sharpen the weather response.
5. **Spatial embeddings.** Learned per-cell embeddings, or a graph/CNN over the H3 grid, would let the model share information between neighbouring cells beyond the three hand-picked anchor distances.
6. **More history.** Extending beyond the current window would stabilise the rare high-demand tail that both models currently under-predict (visible as the RMSE >> MAE gap).

#### 3.8d Operational interpretation (non-technical)

Translated for the client: at the chosen H3 r6 x 4h resolution, the best model predicts the number of pickups in a roughly 14 mi^2 neighbourhood over a 4-hour window with an average error of about 25 trips. In the dense core and at the airports - where a single bucket can hold hundreds of trips - that is a small relative miss and is good enough to plan how many vehicles to pre-position per shift. In quiet outer neighbourhoods, where a bucket holds only a handful of trips, the same 25-trip *average* error is dominated by a few hard-to-predict spikes (RMSE is far larger than MAE), so the model is reliable for *where the demand reliably is*, and weaker on rare surges at the edge.

For the fleet operator the practical message is twofold: (1) demand is predictable enough from calendar, weather and location alone - without ever looking at yesterday's demand - to support shift-level vehicle allocation in the high-demand zones, which is exactly where capital is committed first; and (2) the model should be used as a planning aid for the dense core, with a safety buffer rather than point predictions in the sparse periphery. This feeds directly into the strategic/tactical/operational recommendations consolidated in Section 5.

### Summary and hand-off

- **3.1-3.2 - foundation.** Target `demand_count` on the zero-filled H3 r6 x 4h panel; features = cyclical calendar + weekend flag, cell centroid coordinates, distances to the three demand anchors (downtown, O'Hare, Midway), and weather. Per-bucket trip aggregates and any same/past demand are excluded as leakage. Validation is a day-grouped 15% holdout with 5-fold GroupKFold for selection, a log1p target, and a shared `evaluate` on the original scale.
- **3.3-3.4 - SVM.** A linear SVR baseline (MAE ~66, R^2 ~0.63) is lifted by an RBF kernel with grid-searched C/gamma/epsilon to MAE **30.8** (R^2 0.875, RMSE 119).
- **3.5-3.6 - neural network.** A two-layer feed-forward net reaches the best holdout error (MAE **24.9**, R^2 0.92); tuning over the baseline added almost nothing (baseline MAE 25.8). SHAP shows time-of-day and the hotspot distances as the dominant drivers.
- **3.7 - resolution.** Across H3 r5/r6/r7, Community Area, census tract and 1h/4h/8h/1d bins (compared on scale-free R^2 and nMAE), **H3 r6 x 4h** is the recommended operating point - finer grids and 1h bins are too zero-inflated (r7 `R2` 0.33, census tract 0.08-0.18), daily bins lose the intra-day signal, census tract is also structurally incomplete and not rescalable.
- **3.8 - comparison.** The tuned NN (MAE 24.9) is the only model that beats the naive benchmark (MAE 29.1) and is ~1800x faster to serve than the RBF-SVM (MAE 30.8), which itself does not beat the benchmark. A deep-learning approach is therefore clearly warranted here.

These results inform the Discussion and Outlook in Section 5.